# SeguWatts Vision: treino do detector

Treino customizado de uma **YOLO11 nano** para reconhecer, em cômodos de casas reais:

| Classe | O que é |
|---|---|
| `lampada_acesa` | lâmpada, luminária ou abajur emitindo luz |
| `lampada_apagada` | lâmpada, luminária ou abajur desligado |
| `tela_ligada` | TV ou monitor com imagem |
| `tela_desligada` | TV ou monitor preto |
| `pessoa` | pessoa no cômodo |

**Antes de rodar**

1. *Ambiente de execução → Alterar o tipo → GPU T4*.
2. Em 🔑 *Secrets* (barra lateral), crie `ROBOFLOW_API_KEY` com a chave do Roboflow e libere o acesso deste notebook. A chave nunca fica escrita no notebook.
3. Ajuste `WORKSPACE`, `PROJETO` e `VERSAO` na célula 2.

Tempo total: cerca de 30 a 40 minutos com ~500 imagens.

In [ ]:
# 1. Dependências (mesma versão do requirements.txt do repositório)
!pip install -q ultralytics==8.4.214 roboflow
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
# 2. Dataset anotado no Roboflow (exportado no formato YOLOv8/YOLO11)
from google.colab import userdata
from roboflow import Roboflow

WORKSPACE = "seu-workspace"      # ajuste
PROJETO = "seguwatts-vision"     # ajuste
VERSAO = 1                       # ajuste

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dataset = rf.workspace(WORKSPACE).project(PROJETO).version(VERSAO).download("yolov8")
DATA_YAML = f"{dataset.location}/data.yaml"
print(open(DATA_YAML).read())

# Alternativa sem Roboflow: envie o .zip exportado para o Drive e descompacte:
# from google.colab import drive; drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/seguwatts-vision.zip -d /content/dataset
# DATA_YAML = "/content/dataset/data.yaml"

In [ ]:
# 3. Quantas instâncias de cada classe existem em cada divisão (vai para o relatório)
import collections, glob, os, yaml
import pandas as pd

nomes = yaml.safe_load(open(DATA_YAML))["names"]
nomes = nomes if isinstance(nomes, list) else [nomes[i] for i in sorted(nomes)]
base = os.path.dirname(DATA_YAML)
linhas = []
for divisao in ["train", "valid", "test"]:
    contagem = collections.Counter()
    arquivos = glob.glob(f"{base}/{divisao}/labels/*.txt")
    for arq in arquivos:
        for linha in open(arq):
            if linha.strip():
                contagem[nomes[int(linha.split()[0])]] += 1
    linhas.append({"divisão": divisao, "imagens": len(arquivos), **{n: contagem[n] for n in nomes}})
tabela = pd.DataFrame(linhas).set_index("divisão")
tabela

In [ ]:
# 4. Treino: transfer learning a partir dos pesos COCO
from ultralytics import YOLO

modelo = YOLO("yolo11n.pt")
modelo.train(
    data=DATA_YAML,
    epochs=80,
    patience=20,       # para cedo se a validação parar de melhorar
    imgsz=640,
    batch=16,
    seed=42,           # reprodutível
    project="runs",
    name="seguwatts",
    exist_ok=True,
)

In [ ]:
# 5. Avaliação no conjunto de TESTE (imagens que o modelo nunca viu no treino)
import json

melhor = YOLO("runs/seguwatts/weights/best.pt")
met = melhor.val(data=DATA_YAML, split="test", imgsz=640, plots=True, project="runs", name="teste", exist_ok=True)

por_classe = {
    nomes[c]: {
        "precisao": round(float(met.box.p[i]), 3),
        "recall": round(float(met.box.r[i]), 3),
        "mAP50": round(float(met.box.ap50[i]), 3),
        "mAP50_95": round(float(met.box.ap[i]), 3),
    }
    for i, c in enumerate(met.box.ap_class_index)
}
resumo = {
    "mAP50": round(float(met.box.map50), 3),
    "mAP50_95": round(float(met.box.map), 3),
    "precisao": round(float(met.box.mp), 3),
    "recall": round(float(met.box.mr), 3),
    "por_classe": por_classe,
}
os.makedirs("resultados", exist_ok=True)
json.dump(resumo, open("resultados/metricas_teste.json", "w"), indent=2, ensure_ascii=False)
pd.DataFrame(por_classe).T

In [ ]:
# 6. Curvas de treino, matriz de confusão e curva precisão x recall
from IPython.display import Image, display
for arquivo in ["runs/seguwatts/results.png",
                "runs/teste/confusion_matrix_normalized.png",
                "runs/teste/BoxPR_curve.png"]:
    if os.path.exists(arquivo):
        print(arquivo); display(Image(arquivo, width=900))

In [ ]:
# 7. Exemplos de detecção em imagens de teste
import random, shutil
amostras = random.Random(42).sample(glob.glob(f"{base}/test/images/*"), k=min(8, len(glob.glob(f"{base}/test/images/*"))))
melhor.predict(amostras, imgsz=640, conf=0.5, save=True, project="runs", name="exemplos", exist_ok=True)
for img in sorted(glob.glob("runs/exemplos/*"))[:4]:
    display(Image(img, width=600))

In [ ]:
# 8. FPS em CPU (simula o notebook do morador, sem GPU)
import time, cv2, numpy as np
quadros = [cv2.imread(a) for a in amostras]
for imgsz in (640, 480):
    melhor.predict(quadros[0], imgsz=imgsz, device="cpu", verbose=False)  # aquecimento
    t = time.perf_counter()
    n = 0
    for _ in range(5):
        for q in quadros:
            melhor.predict(q, imgsz=imgsz, device="cpu", verbose=False); n += 1
    print(f"imgsz={imgsz}: {n / (time.perf_counter() - t):.1f} FPS em CPU (Colab)")

In [ ]:
# 9. Empacota pesos e resultados para o repositório
for origem, destino in [
    ("runs/seguwatts/weights/best.pt", "modelo/best.pt"),
    ("runs/seguwatts/results.png", "resultados/curvas_treino.png"),
    ("runs/teste/confusion_matrix_normalized.png", "resultados/matriz_confusao.png"),
    ("runs/teste/BoxPR_curve.png", "resultados/curva_pr.png"),
]:
    if os.path.exists(origem):
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        shutil.copy(origem, destino)
for i, img in enumerate(sorted(glob.glob("runs/exemplos/*"))[:6]):
    shutil.copy(img, f"resultados/exemplo_{i + 1}{os.path.splitext(img)[1]}")
tabela.to_csv("resultados/contagem_classes.csv")
!zip -qr seguwatts_vision_resultados.zip modelo resultados
from google.colab import files
files.download("seguwatts_vision_resultados.zip")

## Depois do treino

1. Descompacte `seguwatts_vision_resultados.zip` na raiz do repositório (pastas `modelo/` e `resultados/`).
2. Rode o detector: `python -m seguwatts_vision.detector --demo`.
3. Copie os números de `resultados/metricas_teste.json` para o relatório (`docs/relatorio.md`).